# Import libraries and load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def show_step(df, step_name):
    print(f"\n===== {step_name} =====")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    display(df.head(3))

processed_path = Path("../data_processed/processed_final_data/processed_final_data.csv")
df = pd.read_csv(processed_path)

show_step(df, "Raw processed_final_data")


===== Raw processed_final_data =====
Shape: (583151, 13)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ


# Convert timestamps to datetime

In [2]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["last_activity"] = pd.to_datetime(df["last_activity"], errors="coerce")

show_step(df, "After converting timestamp & last_activity")



===== After converting timestamp & last_activity =====
Shape: (583151, 13)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ


# Compute resolution time in days

In [3]:
df["resolution_time_days"] = (
    df["last_activity"] - df["timestamp"]
).dt.total_seconds() / (3600 * 24)

show_step(df, "After adding resolution_time_days")



===== After adding resolution_time_days =====
Shape: (583151, 14)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district', 'resolution_time_days']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district,resolution_time_days
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ,274.113254
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ,274.725701
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ,328.909908


# Define "done" flag

In [4]:
DONE_STATE1 = "เสร็จสิ้น"
DONE_STATE2 = "แก้ไขแล้วเสร็จ"
df["is_done"] = df["state"].isin([DONE_STATE1, DONE_STATE2]).astype(int)


show_step(df, "After creating is_done (1=เสร็จสิ้น)")
print(df["is_done"].value_counts())



===== After creating is_done (1=เสร็จสิ้น) =====
Shape: (583151, 15)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district', 'resolution_time_days', 'is_done']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district,resolution_time_days,is_done
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ,274.113254,1
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ,274.725701,1
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ,328.909908,1


is_done
1    475813
0    107338
Name: count, dtype: int64


# Clean weird / invalid times

In [5]:
print("Before filtering:", df.shape)

df = df.dropna(subset=["timestamp", "last_activity", "resolution_time_days"])
df = df[(df["resolution_time_days"] >= 0) & (df["resolution_time_days"] <= 365)]

print("After filtering:", df.shape)
show_step(df, "After filtering invalid resolution_time_days")


Before filtering: (583151, 15)
After filtering: (548411, 15)

===== After filtering invalid resolution_time_days =====
Shape: (548411, 15)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district', 'resolution_time_days', 'is_done']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district,resolution_time_days,is_done
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ,274.113254,1
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ,274.725701,1
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ,328.909908,1


# Build time_score

In [6]:
mask_done = df["is_done"] == 1

conditions_time = [
    mask_done & (df["resolution_time_days"] <= 3),
    mask_done & (df["resolution_time_days"] > 3) & (df["resolution_time_days"] <= 7),
    mask_done & (df["resolution_time_days"] > 7) & (df["resolution_time_days"] <= 30),
    mask_done & (df["resolution_time_days"] > 30),
]

choices_time = [3, 2, 1, 0]

df["time_score"] = np.select(conditions_time, choices_time, default=0)

show_step(df, "After creating time_score")
print(df["time_score"].value_counts().sort_index())



===== After creating time_score =====
Shape: (548411, 16)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district', 'resolution_time_days', 'is_done', 'time_score']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district,resolution_time_days,is_done,time_score
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ,274.113254,1,0
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ,274.725701,1,0
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ,328.909908,1,0


time_score
0    235350
1     89575
2     66691
3    156795
Name: count, dtype: int64


# Build star_score

In [7]:
# Ensure star is numeric
df["star"] = pd.to_numeric(df["star"], errors="coerce")

conditions_star = [
    df["star"] >= 4,                # good rating
    (df["star"] >= 2) & (df["star"] <= 3),  # neutral
    (df["star"] >= 0) & (df["star"] <= 1),  # bad
]

choices_star = [1, 0, -1]

df["star_score"] = np.select(conditions_star, choices_star, default=0)

show_step(df, "After creating star_score")
print(df["star_score"].value_counts().sort_index())



===== After creating star_score =====
Shape: (548411, 17)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district', 'resolution_time_days', 'is_done', 'time_score', 'star_score']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district,resolution_time_days,is_done,time_score,star_score
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ,274.113254,1,0,0
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ,274.725701,1,0,1
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ,328.909908,1,0,0


star_score
-1     36081
 0    349511
 1    162819
Name: count, dtype: int64


# Build reopen_score

In [8]:
df["count_reopen"] = pd.to_numeric(df["count_reopen"], errors="coerce").fillna(0)

conditions_reopen = [
    df["count_reopen"] >= 1,   # reopened at least once
]

choices_reopen = [-1]

df["reopen_score"] = np.select(conditions_reopen, choices_reopen, default=0)

show_step(df, "After creating reopen_score")
print(df["reopen_score"].value_counts().sort_index())



===== After creating reopen_score =====
Shape: (548411, 18)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district', 'resolution_time_days', 'is_done', 'time_score', 'star_score', 'reopen_score']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district,resolution_time_days,is_done,time_score,star_score,reopen_score
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ,274.113254,1,0,0,0
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ,274.725701,1,0,1,0
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ,328.909908,1,0,0,0


reopen_score
-1     39551
 0    508860
Name: count, dtype: int64


# Combine into efficiency_score (0–5)

In [9]:
# Base = 1 if done, else 0
base_done = df["is_done"]

df["efficiency_raw"] = (
    base_done
    + df["time_score"]
    + df["star_score"]
    + df["reopen_score"]
)

# Clip to [0, 5]
df["efficiency_score"] = df["efficiency_raw"].clip(lower=0, upper=5).astype(int)

show_step(df, "After creating efficiency_score")
print(df["efficiency_score"].value_counts().sort_index())



===== After creating efficiency_score =====
Shape: (548411, 20)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'district', 'resolution_time_days', 'is_done', 'time_score', 'star_score', 'reopen_score', 'efficiency_raw', 'efficiency_score']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,district,resolution_time_days,is_done,time_score,star_score,reopen_score,efficiency_raw,efficiency_score
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,"100.53084,13.81865",กรุงเทพมหานคร,2021-09-03 19:51:09.453003,เสร็จสิ้น,-1,0,2022-06-04 22:34:14.609206,13.81865,100.53084,บางซื่อ,274.113254,1,0,0,0,1,1
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.66709,13.67891",กรุงเทพมหานคร,2021-09-19 21:56:08.924992,เสร็จสิ้น,4,0,2022-06-21 15:21:09.532782,13.67891,100.66709,ประเวศ,274.725701,1,0,1,0,2,2
2,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ","100.53099,13.81853",กรุงเทพมหานคร,2021-10-14 17:45:27.713884,เสร็จสิ้น,-1,0,2022-09-08 15:35:43.784519,13.81853,100.53099,บางซื่อ,328.909908,1,0,0,0,1,1


efficiency_score
0    120918
1     93441
2     88763
3     75811
4    105627
5     63851
Name: count, dtype: int64


# Save for ML notebook

In [10]:
target_col = "efficiency_score"

feature_cols = [
    "type",
    "district",
    "province",
    "lat",
    "lon",
    "timestamp",      
]

X = df[feature_cols].copy()
y = df[target_col].copy()

X.to_csv("../data_processed/X_features.csv", index=False)
y.to_csv("../data_processed/y_target.csv", index=False)
df.to_csv("../data_processed/df_ml_base.csv", index=False)

print("Saved X_features.csv, y_target.csv, df_ml_base.csv")


Saved X_features.csv, y_target.csv, df_ml_base.csv
